# Brainbuilder · Research explorer

A read-only view of the captured evidence ledger. This notebook uses only Python standard-library modules. Jupyter is optional for running it interactively. Run the CLI to initialize and seed the database first.

**Interpretation:** distinct captured threads describe this exploratory corpus. They are not representative demand, unique people, paying customers, or learning-effectiveness estimates. Vendor claims and primary learning studies stay separate from customer observations.


In [ ]:
from pathlib import Path
import json
import sqlite3
from datetime import date

AS_OF = "2026-11-07"  # Set the actual review date for each repeated study.
STALE_DAYS = 90
ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "bb_research.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the workbench directory.")
DB = ROOT / "data" / "research.sqlite"
if not DB.exists():
    raise RuntimeError("Initialize and import evidence with bb_research.py before running this notebook.")
conn = sqlite3.connect(DB.resolve().as_uri() + "?mode=ro", uri=True)
conn.row_factory = sqlite3.Row

def records(sql, params=()):
    return [dict(row) for row in conn.execute(sql, params)]

def show(value):
    print(json.dumps(value, indent=2, ensure_ascii=False))

print("Read-only database:", DB)
print("Report date:", AS_OF)


## Evidence kinds and source-check coverage

A source check means the observation was checked against content at the recorded access depth. It does not validate an anonymous self-report. An assistant check is not a human check.


In [ ]:
show(records("""
SELECT evidence_kind, review_status, reviewer_type, content_access,
       COUNT(*) AS observations,
       COUNT(DISTINCT thread_key) AS captured_threads
FROM evidence
GROUP BY evidence_kind, review_status, reviewer_type, content_access
ORDER BY evidence_kind, review_status, content_access
"""))


## Problems in customer observations

Rejected observations and all other evidence kinds are excluded. A thread counts once per problem. The same thread can occur in multiple problem rows, so row totals must not be summed as independent demand.


In [ ]:
show(records("""
SELECT COALESCE(NULLIF(problem, ''), 'Uncoded') AS problem,
       COUNT(DISTINCT thread_key) AS captured_customer_threads,
       COUNT(DISTINCT CASE WHEN review_status = 'verified' THEN thread_key END) AS source_checked_threads,
       COUNT(DISTINCT CASE WHEN review_status = 'verified' AND reviewer_type = 'human' THEN thread_key END) AS human_checked_threads,
       COUNT(DISTINCT CASE WHEN content_access = 'full_text' THEN thread_key END) AS full_text_threads,
       COUNT(*) AS observations
FROM evidence
WHERE evidence_kind = 'customer_signal' AND review_status != 'rejected'
GROUP BY problem
ORDER BY captured_customer_threads DESC, problem
"""))


## Counterevidence, workarounds, and payment observations

Read the actual text. A nonempty payment field may say that no budget was stated; do not count it as willingness to pay. A viable existing workaround can challenge the need to build.


In [ ]:
show(records("""
SELECT evidence_id, title, source_url, observation, workaround,
       willingness_to_pay, negative_evidence, inference
FROM evidence
WHERE evidence_kind = 'customer_signal' AND review_status != 'rejected'
ORDER BY problem, evidence_id
"""))


## Capture freshness and publication uncertainty

The flag is about capture age, not the scientific validity of an older paper. Abstracts, snippets, and unknown publication dates remain visible.


In [ ]:
anchor = date.fromisoformat(AS_OF)
freshness = []
for row in records("SELECT evidence_id, evidence_kind, retrieved_at, published_at, content_access FROM evidence"):
    row["capture_age_days"] = (anchor - date.fromisoformat(row["retrieved_at"])).days
    row["needs_refresh"] = row["capture_age_days"] > STALE_DAYS
    row["future_capture_date"] = row["capture_age_days"] < 0
    freshness.append(row)
show(freshness)


## Hypotheses and evidence relationships

A hypothesis state is a researcher decision; it is never automatically upgraded from source counts. The next table counts only non-rejected customer threads by relationship. Competitor and learning sources may still be linked as context in the database.


In [ ]:
show(records("SELECT hypothesis_id, statement, status, decision_rule, next_test FROM hypotheses ORDER BY hypothesis_id"))
show(records("""
SELECT h.hypothesis_id, l.relationship,
       COUNT(DISTINCT e.thread_key) AS captured_customer_threads
FROM hypotheses h
JOIN evidence_hypotheses l USING (hypothesis_id)
JOIN evidence e USING (evidence_id)
WHERE e.evidence_kind = 'customer_signal' AND e.review_status != 'rejected'
GROUP BY h.hypothesis_id, l.relationship
ORDER BY h.hypothesis_id, l.relationship
"""))


## Search coverage and audit gaps

`null` means unknown. Do not turn unknown counts into zero or sum included threads across overlapping searches. Planned searches are not performed work.


In [ ]:
show(records("""
SELECT performed_at, channel, query, status, results_returned,
       results_reviewed, included_threads, notes, reviewer
FROM query_log ORDER BY performed_at, query_id
"""))
show(records("""
SELECT evidence_id, title, source_url, review_status, content_access, provenance_notes
FROM evidence
WHERE review_status != 'verified' OR content_access IN ('snippet', 'abstract', 'metadata', 'not_reviewed')
ORDER BY review_status, evidence_id
"""))
show(records("SELECT COUNT(*) AS evidence_revisions FROM evidence_revisions"))


## Continue with the report

Use the CLI to write a new dated report after importing new, source-checked evidence. This notebook has made no source requests or database writes.

```bash
python3 bb_research.py export --as-of YYYY-MM-DD --out reports/YYYY-MM-DD
```


In [ ]:
report_path = ROOT / "reports" / AS_OF / "report.html"
print("Expected dated report:", report_path)
print("Report exists:", report_path.is_file())
conn.close()
